In [1]:
from curator.simulate import MLCalculator
from curator.simulate.core.simulator import Simulator

from curator.simulate.callbacks.thermo_uncertainty import ThermoWithUncertainty
from curator.simulate.callbacks.thermo_uncertainty import ThermoWithUncertainty
from ase.md.langevin import Langevin
from ase.io import read, write

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/opt/conda/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


In [2]:
# it is possible to directly use a trained model to generate an ASE calculator
atoms = read('../example/LiFePO4.traj')
calc = MLCalculator('../example/train/model_path')

# After assigning calculator to the atoms, you can calculate many properties with ASE
atoms.calc = calc
atoms.get_potential_energy()

/workspace/curator/curator/utils.py:91: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  obj = torch.load(model_file, map_location=torch.device(device))


-179.77609252929688

You need to create a engine to run the desired simulation. The engine can be ase's dynamics or other engines. It is also possible to define a engine by yourself through inheriting the base `Engine` class

In [3]:
from curator.simulate.engines.ase_md import MDEngine
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

MaxwellBoltzmannDistribution(atoms, temperature_K=300)
# dyn = Langevin(atoms, timestep=0.5, friction=0.2, temperature_K=300)  # not recommended to create engine in this way
engine = MDEngine('ase.md.langevin.Langevin', timestep=0.5, friction=0.2, temperature_K=300)
# engine.setup(atoms)
# engine.run(100)

Although a simple `Engine` class is capable of some simple simulation tasks, it is inconvenient to do some pre-process and post-process for the atoms, calculator, and also logging some per-step simulation information.

That's why we created the `Simulator` class, which works in a way alike to [pytorch lightning](https://lightning.ai/docs/pytorch/stable/).

You can easily add some callbacks into a simulator. These callbacks is very flexible and powerful for different usages.

Here we will showcase some useful callbacks which are able to monitor a MD simulation, save structures when simulating, and pre-process the initial structures.

In [4]:
from curator.simulate.callbacks import MDThermoLogger
from curator.simulate.callbacks import CalculatorAssign

In [5]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,   # output log to screen
    format="%(asctime)s - %(levelname)s - %(message)s"
)

thermo_logger = MDThermoLogger()       # define a default logger
calc_cb = CalculatorAssign(calc)       # assign calculator
simulator = Simulator(
    init_traj='../example/LiFePO4.traj',
    engine=engine,
    callbacks=[thermo_logger, calc_cb],
)

In [6]:
simulator.run(100)

2025-11-28 17:54:19,626 - INFO -            step           epot           ekin           etot
2025-11-28 17:54:19,854 - INFO - Calcator assigned to atoms.
2025-11-28 17:54:19,856 - INFO -               1     -183.01427        1.75319     -181.26108
2025-11-28 17:54:20,159 - INFO -               2     -181.51332        0.74264     -180.77068
2025-11-28 17:54:20,177 - INFO -               3     -182.85933        1.66323     -181.19610
2025-11-28 17:54:20,194 - INFO -               4     -182.89337        1.63463     -181.25874
2025-11-28 17:54:20,210 - INFO -               5     -182.16423        1.04137     -181.12286
2025-11-28 17:54:20,227 - INFO -               6     -182.41759        1.19012     -181.22747
2025-11-28 17:54:20,245 - INFO -               7     -183.19505        1.80156     -181.39349
2025-11-28 17:54:20,262 - INFO -               8     -182.90762        1.45110     -181.45652
2025-11-28 17:54:20,279 - INFO -               9     -182.72971        0.99125     -181.73846

In most cases, we may care about the uncertainties of the structures generated in a simulation. We can add an uncertainty callback to achieve that.

In [7]:
from curator.simulate.uncertainty import MahalanobisUncertainty

maha = MahalanobisUncertainty(calculator=calc, dataset='../example/LiFePO4.traj')

2025-11-28 17:54:30,289 - INFO - Calculating features from provided dataset <../example/LiFePO4.traj>.
2025-11-28 17:54:30,291 - INFO - Calculating precision matrix from 2257 structures.
2025-11-28 17:54:30,647 - INFO - Processing batch 1/18


/opt/conda/lib/python3.11/site-packages/torch/nn/modules/module.py:1842: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


2025-11-28 17:54:31,384 - INFO - Processing batch 2/18
2025-11-28 17:54:32,110 - INFO - Processing batch 3/18
2025-11-28 17:54:32,845 - INFO - Processing batch 4/18
2025-11-28 17:54:33,556 - INFO - Processing batch 5/18
2025-11-28 17:54:34,263 - INFO - Processing batch 6/18
2025-11-28 17:54:34,975 - INFO - Processing batch 7/18
2025-11-28 17:54:35,689 - INFO - Processing batch 8/18
2025-11-28 17:54:36,398 - INFO - Processing batch 9/18
2025-11-28 17:54:37,108 - INFO - Processing batch 10/18
2025-11-28 17:54:37,846 - INFO - Processing batch 11/18
2025-11-28 17:54:38,566 - INFO - Processing batch 12/18
2025-11-28 17:54:39,307 - INFO - Processing batch 13/18
2025-11-28 17:54:40,093 - INFO - Processing batch 14/18
2025-11-28 17:54:40,850 - INFO - Processing batch 15/18
2025-11-28 17:54:41,598 - INFO - Processing batch 16/18
2025-11-28 17:54:42,336 - INFO - Processing batch 17/18
2025-11-28 17:54:42,930 - INFO - Processing batch 18/18


In [9]:
from curator.simulate.callbacks import ThermoWithUncertainty
maha_thermo = ThermoWithUncertainty(uncertainty_backend=maha)

In [10]:
simulator.callbacks = [maha_thermo, calc_cb]

In [12]:
simulator.run(1000)

2025-11-28 17:56:05,247 - INFO -            step           epot           ekin           etotmahalanobis_distance     is_outlier     is_warning
2025-11-28 17:56:05,248 - INFO - Calcator assigned to atoms.
2025-11-28 17:56:05,270 - INFO -             202     -182.98621        0.86651     -182.11969       15.82292        0.00000        0.00000
2025-11-28 17:56:05,290 - INFO -             203     -183.14532        0.94410     -182.20122       15.73689        0.00000        0.00000
2025-11-28 17:56:05,309 - INFO -             204     -183.10590        0.99703     -182.10886       15.82308        0.00000        0.00000
2025-11-28 17:56:05,327 - INFO -             205     -183.02209        0.83541     -182.18668       15.75465        0.00000        0.00000
2025-11-28 17:56:05,345 - INFO -             206     -182.97923        0.91528     -182.06396       15.88885        0.00000        0.00000
2025-11-28 17:56:05,364 - INFO -             207     -183.06113        1.01210     -182.04903       

In [3]:
import torch

In [5]:
a =torch.randn(3584, 129)
b = torch.randn(129, 500)

In [6]:
a @ b

tensor([[ -1.0177, -25.5826,   2.7012,  ..., -17.2837,  -5.3474,  16.9933],
        [  8.8602,  17.3571,  -8.3821,  ...,  -7.4742,  -7.0939,  -3.9934],
        [-21.1038,  -6.9150,  -1.2727,  ...,  29.9888,  17.3085, -22.0121],
        ...,
        [ 14.0875,   5.7641,  -2.1081,  ...,   5.6166,  -4.5089,  -4.0875],
        [ 10.7391, -15.1346, -17.5670,  ..., -16.2054,   6.8459,  10.7584],
        [  9.1608,   6.8650,  15.4067,  ...,   0.8749, -19.0322, -19.9643]])

In [6]:
a = torch.randn(129)

In [8]:
a.sum().item()

12.87747859954834

In [4]:
maha = MahalanobisUncertainty(calculator=calc, dataset='../example/LiFePO4.traj')

2025-11-28 17:46:48,574 - INFO - Calculating features from provided dataset <../example/LiFePO4.traj>.
2025-11-28 17:46:48,576 - INFO - Calculating precision matrix from 2257 structures.
2025-11-28 17:46:48,978 - INFO - Processing batch 1/18


/opt/conda/lib/python3.11/site-packages/torch/nn/modules/module.py:1842: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


2025-11-28 17:46:50,204 - INFO - Processing batch 2/18
2025-11-28 17:46:51,109 - INFO - Processing batch 3/18
2025-11-28 17:46:52,093 - INFO - Processing batch 4/18
2025-11-28 17:46:52,806 - INFO - Processing batch 5/18
2025-11-28 17:46:53,522 - INFO - Processing batch 6/18
2025-11-28 17:46:54,235 - INFO - Processing batch 7/18
2025-11-28 17:46:54,948 - INFO - Processing batch 8/18
2025-11-28 17:46:55,661 - INFO - Processing batch 9/18
2025-11-28 17:46:56,372 - INFO - Processing batch 10/18
2025-11-28 17:46:57,086 - INFO - Processing batch 11/18
2025-11-28 17:46:57,794 - INFO - Processing batch 12/18
2025-11-28 17:46:58,502 - INFO - Processing batch 13/18
2025-11-28 17:46:59,216 - INFO - Processing batch 14/18
2025-11-28 17:46:59,932 - INFO - Processing batch 15/18
2025-11-28 17:47:00,645 - INFO - Processing batch 16/18
2025-11-28 17:47:01,357 - INFO - Processing batch 17/18
2025-11-28 17:47:01,932 - INFO - Processing batch 18/18


In [12]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,   # output log to screen
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [5]:
maha.high_threshold

28.501530647277836

In [6]:
from ase.io import Trajectory

In [7]:
traj = Trajectory('../example/LiFePO4.traj')

In [6]:
import torch
from torch.utils.data import DataLoader
from curator.data import AseDataset, collate_atomsdata

In [9]:
dataset = AseDataset('../example/LiFePO4.traj')
dataloader = DataLoader(dataset, batch_size=10, shuffle=True, collate_fn=collate_atomsdata)
for i, batch in enumerate(dataloader):
    print(batch)
    if i > 3:
        break

{'n_atoms': tensor([28, 28, 28, 28, 28, 28, 28, 28, 28, 28]), 'atomic_numbers': tensor([26, 26, 26, 26,  3,  3,  3,  3,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,
         8,  8,  8,  8,  8,  8, 15, 15, 15, 15, 26, 26, 26, 26,  3,  3,  3,  3,
         8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8, 15, 15,
        15, 15, 26, 26, 26, 26,  3,  3,  3,  3,  8,  8,  8,  8,  8,  8,  8,  8,
         8,  8,  8,  8,  8,  8,  8,  8, 15, 15, 15, 15, 26, 26, 26, 26,  3,  3,
         3,  3,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,
        15, 15, 15, 15, 26, 26, 26, 26,  3,  3,  3,  3,  8,  8,  8,  8,  8,  8,
         8,  8,  8,  8,  8,  8,  8,  8,  8,  8, 15, 15, 15, 15, 26, 26, 26, 26,
         3,  3,  3,  3,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,
         8,  8, 15, 15, 15, 15, 26, 26, 26, 26,  3,  3,  3,  3,  8,  8,  8,  8,
         8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8, 15, 15, 15, 15, 26, 26,
        26, 26,  3,  3,  3,  3,  8,  8, 

In [13]:
next(calc.model.parameters()).device

device(type='cuda', index=0)

In [11]:
maha(atoms)

/opt/conda/lib/python3.11/site-packages/torch/nn/modules/module.py:1842: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [19]:
atoms.get_potential_energy()

-182.91700744628906

In [20]:
MaxwellBoltzmannDistribution(atoms, temperature_K=300)
dyn = Langevin(atoms, timestep=0.5, friction=0.2, temperature_K=300)

In [27]:
dyn.run(100)

True

In [28]:
atoms.get_potential_energy()

-182.99713134765625

In [14]:
simulator.run(steps=100)

2025-11-20 15:03:15,722 - INFO -            step           epot           ekin           etot
2025-11-20 15:03:15,745 - INFO -               1     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,762 - INFO -               2     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,779 - INFO -               3     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,795 - INFO -               4     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,811 - INFO -               5     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,827 - INFO -               6     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,843 - INFO -               7     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,859 - INFO -               8     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,876 - INFO -               9     -183.65689        0.63391     -183.02298
2025-11-20 15:03:15,892 - INFO -              10     -183.65

In [ ]:
import 

In [ ]:
from typing import Any

In [ ]:
isinstance(_resolve('ase.md.langevin.Langevin'), type)